In [0]:
results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)
races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
qualifying = spark.read.csv("/Volumes/gr5069/raw/f1_data/qualifying.csv", header=True, inferSchema=True)

display(results)
display(races)
display(qualifying)


In [0]:
print(results.columns)
print(races.columns)
print(qualifying.columns)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import os
import tempfile

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [0]:
results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True, inferSchema=True)
races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True, inferSchema=True)
qualifying = spark.read.csv("/Volumes/gr5069/raw/f1_data/qualifying.csv", header=True, inferSchema=True)

display(results)
display(races)
display(qualifying)

In [0]:
df = (
    results.select("raceId", "driverId", "constructorId", "grid", "positionOrder", "points")
    .join(
        races.select("raceId", "year", "round", "circuitId"),
        on="raceId",
        how="left"
    )
    .join(
        qualifying.select("raceId", "driverId", "constructorId", "position")
                  .withColumnRenamed("position", "qualifying_position"),
        on=["raceId", "driverId", "constructorId"],
        how="left"
    )
)

display(df)

In [0]:
pdf = df.toPandas()

pdf["target"] = (pdf["points"] > 0).astype(int)

# fill missing qualifying positions with a large number
pdf["qualifying_position"] = pdf["qualifying_position"].fillna(20)

# keep only needed columns
model_df = pdf[
    [
        "grid",
        "driverId",
        "constructorId",
        "year",
        "round",
        "circuitId",
        "qualifying_position",
        "target"
    ]
].dropna()

print(model_df.shape)
model_df.head()

In [0]:
X = model_df.drop(columns="target")
y = model_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

In [0]:
mlflow.set_experiment("/Users/tl2949@columbia.edu/f1_homework4_mlflow")

In [0]:
param_grid = [
    {"n_estimators": 50,  "max_depth": 5,  "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 5,  "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 150, "max_depth": 5,  "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 50,  "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 150, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 50,  "max_depth": 15, "min_samples_split": 4, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 15, "min_samples_split": 4, "min_samples_leaf": 1},
    {"n_estimators": 150, "max_depth": 15, "min_samples_split": 4, "min_samples_leaf": 2},
    {"n_estimators": 200, "max_depth": 20, "min_samples_split": 5, "min_samples_leaf": 2},
]

In [0]:
all_runs = []

for i, params in enumerate(param_grid, start=1):
    with mlflow.start_run(run_name=f"rf_run_{i}"):
        
        # log parameters
        mlflow.log_params(params)
        
        # build model
        model = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42
        )
        
        model.fit(X_train, y_train)
        
        # predictions
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
        # metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_prob)
        
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("roc_auc", roc_auc)
        
        # log model
        mlflow.sklearn.log_model(model, "model")
        
        with tempfile.TemporaryDirectory() as tmpdir:
            # artifact 1: confusion matrix plot
            cm = confusion_matrix(y_test, y_pred)
            disp = ConfusionMatrixDisplay(confusion_matrix=cm)
            disp.plot()
            cm_path = os.path.join(tmpdir, f"confusion_matrix_run_{i}.png")
            plt.savefig(cm_path, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(cm_path)
            
            # artifact 2: feature importance plot
            feat_imp = pd.DataFrame({
                "feature": X.columns,
                "importance": model.feature_importances_
            }).sort_values("importance", ascending=False)
            
            plt.figure(figsize=(8, 5))
            plt.bar(feat_imp["feature"], feat_imp["importance"])
            plt.xticks(rotation=45)
            plt.title(f"Feature Importance Run {i}")
            fi_path = os.path.join(tmpdir, f"feature_importance_run_{i}.png")
            plt.savefig(fi_path, bbox_inches="tight")
            plt.close()
            mlflow.log_artifact(fi_path)
            
            # artifact 3: predictions csv
            pred_df = pd.DataFrame({
                "actual": y_test.values,
                "predicted": y_pred,
                "probability": y_prob
            })
            pred_path = os.path.join(tmpdir, f"predictions_run_{i}.csv")
            pred_df.to_csv(pred_path, index=False)
            mlflow.log_artifact(pred_path)
            
            # artifact 4: metrics csv
            metrics_df = pd.DataFrame([{
                "run": i,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "roc_auc": roc_auc
            }])
            metrics_path = os.path.join(tmpdir, f"metrics_run_{i}.csv")
            metrics_df.to_csv(metrics_path, index=False)
            mlflow.log_artifact(metrics_path)
        
        all_runs.append({
            "run": i,
            **params,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "roc_auc": roc_auc
        })

summary_df = pd.DataFrame(all_runs)
summary_df = summary_df.sort_values("f1", ascending=False)

display(summary_df)

In [0]:
best_run = summary_df.iloc[0]
print("Best run:")
print(best_run)

In [0]:
summary_df.to_csv("/tmp/all_runs_summary.csv", index=False)
print("Saved summary to /tmp/all_runs_summary.csv")

I selected the best model based on the F1 score because this is a binary classification problem and F1 provides a balanced evaluation of both precision and recall. Among the 10 runs, Run 9 performed the best with an F1 score of about 0.604, while also maintaining solid accuracy of 0.782 and a strong ROC-AUC of 0.844. This means the model was not only reasonably accurate overall, but also performed well in distinguishing between drivers who scored points and those who did not. I also reviewed the logged artifacts, including the confusion matrix and feature importance plot, to confirm that the model behaved consistently. Therefore, I selected Run 9 as the best overall model.